# Lab 10: Word Embeddings and Sentiment Analysis

## Q1: Preprocessing and sentence representations

### Q1(a): Preprocessing and stratified samples

In [1]:
from pathlib import Path
import gc
import re
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from IPython.display import display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

LAB10_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if path.name == "Lab 10" and (path / "data").is_dir()),
    None,
)
if LAB10_DIR is None:
    raise FileNotFoundError("Could not find the Lab 10 directory.")
DATA_DIR = LAB10_DIR / "data"
EMBEDDING_PATHS = {
    "Word2Vec": DATA_DIR / "word2vec-google-news-300.model",
    "GloVe": DATA_DIR / "glove.6B.300d.txt",
    "FastText": DATA_DIR / "wiki-news-300d-1M-subword.vec",
}
OUTPUT_DIR = LAB10_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANALOGY_PATH = DATA_DIR / "questions-words.txt"
EMBEDDING_URLS = {
    "Word2Vec": "https://huggingface.co/fse/word2vec-google-news-300/resolve/main/word2vec-google-news-300.model",
    "GloVe": "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
    "FastText": "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip",
}

def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        urllib.request.urlretrieve(url, destination)

def prepare_embedding_data():
    download_file(
        "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt",
        ANALOGY_PATH,
    )
    download_file(EMBEDDING_URLS["Word2Vec"], EMBEDDING_PATHS["Word2Vec"])
    download_file(
        "https://huggingface.co/fse/word2vec-google-news-300/resolve/main/word2vec-google-news-300.model.vectors.npy",
        DATA_DIR / "word2vec-google-news-300.model.vectors.npy",
    )
    glove_archive = DATA_DIR / "glove.6B.zip"
    if not EMBEDDING_PATHS["GloVe"].is_file():
        download_file(EMBEDDING_URLS["GloVe"], glove_archive)
        with zipfile.ZipFile(glove_archive, "r") as archive:
            archive.extract("glove.6B.300d.txt", DATA_DIR)
    fasttext_archive = DATA_DIR / "wiki-news-300d-1M.vec.zip"
    if not EMBEDDING_PATHS["FastText"].is_file():
        download_file(EMBEDDING_URLS["FastText"], fasttext_archive)
        with zipfile.ZipFile(fasttext_archive, "r") as archive:
            archive.extract("wiki-news-300d-1M.vec", DATA_DIR)
        (DATA_DIR / "wiki-news-300d-1M.vec").replace(EMBEDDING_PATHS["FastText"])

def stratified_sample_csv(path, sample_size, seed=42, chunksize=100_000):
    if sample_size % 2:
        raise ValueError("The sample size must be even for the balanced dataset.")
    quota = sample_size // 2
    reservoirs = {}
    rng = np.random.default_rng(seed)

    for chunk in pd.read_csv(
        path,
        header=None,
        usecols=[0, 2],
        dtype={0: "int8", 2: "string"},
        keep_default_na=False,
        chunksize=chunksize,
    ):
        chunk.columns = ["label", "review"]
        for label, group in chunk.groupby("label", sort=False):
            label = int(label)
            if label not in (1, 2):
                raise ValueError(f"Unexpected class label: {label}")
            group = group.assign(_key=rng.random(len(group)))
            candidates = (
                pd.concat([reservoirs[label], group], ignore_index=True)
                if label in reservoirs
                else group
            )
            reservoirs[label] = candidates.nsmallest(quota, "_key")

    if any(len(reservoirs.get(label, [])) != quota for label in (1, 2)):
        raise ValueError(f"{path.name} does not contain enough examples of both classes.")
    sample = pd.concat([reservoirs[1], reservoirs[2]], ignore_index=True)
    return sample.drop(columns="_key").sample(frac=1, random_state=seed).reset_index(drop=True)

def preprocess(text):
    text = re.sub(r"[^a-z\s]", " ", str(text).lower())
    return [word for word in re.findall(r"[a-z]+", text) if word not in ENGLISH_STOP_WORDS]

prepare_embedding_data()
train_df = stratified_sample_csv(DATA_DIR / "train.csv", 10_000)
test_df = stratified_sample_csv(DATA_DIR / "test.csv", 1_000)
train_df.to_csv(OUTPUT_DIR / "q1_stratified_train.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "q1_stratified_test.csv", index=False)
train_tokens = train_df["review"].map(preprocess).tolist()
test_tokens = test_df["review"].map(preprocess).tolist()
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

display(pd.DataFrame({
    "split": ["train", "test"],
    "samples": [len(train_df), len(test_df)],
    "class_1": [(y_train == 1).sum(), (y_test == 1).sum()],
    "class_2": [(y_train == 2).sum(), (y_test == 2).sum()],
}))


,split,samples,class_1,class_2
0,train,10000,5000,5000
1,test,1000,500,500


### Q1(b): Sentence representations using Word2Vec, GloVe, and FastText

In [2]:
q1_representations = {}
def load_embeddings(name):
    path = EMBEDDING_PATHS[name]
    if not path.is_file():
        raise FileNotFoundError(f"Place the {name} embedding file at: {path}")
    if name == "Word2Vec":
        return KeyedVectors.load(str(path), mmap="r")
    if name == "GloVe":
        return KeyedVectors.load_word2vec_format(path, binary=False, no_header=True)
    return KeyedVectors.load_word2vec_format(path, binary=False)

def sentence_vectors(tokenized_sentences, keyed_vectors, tfidf_matrix=None, vocabulary=None):
    result = np.zeros((len(tokenized_sentences), keyed_vectors.vector_size), dtype=np.float32)
    for row_index, tokens in enumerate(tokenized_sentences):
        weights = None
        if tfidf_matrix is not None:
            row = tfidf_matrix.getrow(row_index)
            weights = dict(zip(row.indices, row.data))
        total_weight = 0.0
        for token in tokens:
            if token not in keyed_vectors.key_to_index:
                continue
            weight = 1.0 if weights is None else weights.get(vocabulary.get(token, -1), 0.0)
            if weight:
                result[row_index] += keyed_vectors[token] * weight
                total_weight += weight
        if total_weight:
            result[row_index] /= total_weight
    return result

for model_name in EMBEDDING_PATHS:
    vectors = load_embeddings(model_name)
    train_vectors = sentence_vectors(train_tokens, vectors)
    test_vectors = sentence_vectors(test_tokens, vectors)
    q1_representations[model_name] = (train_vectors, test_vectors)
    np.savez(
        OUTPUT_DIR / f"q1_{model_name.lower()}_sentence_vectors.npz",
        train_vectors=train_vectors,
        test_vectors=test_vectors,
    )
    print(model_name, train_vectors.shape, test_vectors.shape)
    del vectors
    gc.collect()


Word2Vec (10000, 300) (1000, 300)


GloVe (10000, 300) (1000, 300)


FastText (10000, 300) (1000, 300)


## Q2: Word analogies

### Q2(a): Analogy examples

In [3]:
examples = [
    ("good", "bad", "excellent"),
    ("delivery", "late", "fast"),
]

analogy_examples = []
for model_name in EMBEDDING_PATHS:
    vectors = load_embeddings(model_name)
    row = {"embedding": model_name}
    for positive, negative, other_positive in examples:
        words = [positive, negative, other_positive]
        if all(word in vectors.key_to_index for word in words):
            answer = vectors.most_similar(
                positive=[other_positive, negative],
                negative=[positive],
                topn=1,
            )[0][0]
        else:
            answer = "word not in vocabulary"
        row[f"{positive}:{negative}:{other_positive}"] = answer
    analogy_examples.append(row)
    del vectors
    gc.collect()

pd.DataFrame(analogy_examples).to_csv(
    OUTPUT_DIR / "q2_analogy_examples.csv",
    index=False,
)
display(pd.DataFrame(analogy_examples))


,embedding,good:bad:excellent,delivery:late:fast
0,Word2Vec,atrocious,early
1,GloVe,mediocre,early
2,FastText,superb,early


### Q2(b): Analogy dataset accuracy

## Q3: Single-hidden-layer neural networks

### Q3(a): Average word-vector representations

### Q3(b): TF-IDF-weighted word-vector representations

### Q3(c): Macro F1 comparison

In [4]:
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
)
train_tfidf = tfidf_vectorizer.fit_transform([" ".join(tokens) for tokens in train_tokens])
test_tfidf = tfidf_vectorizer.transform([" ".join(tokens) for tokens in test_tokens])
tfidf_vocabulary = tfidf_vectorizer.vocabulary_

network_results = []
for model_name in EMBEDDING_PATHS:
    vectors = load_embeddings(model_name)
    representations = {
        "Average": (
            sentence_vectors(train_tokens, vectors),
            sentence_vectors(test_tokens, vectors),
        ),
        "TF-IDF weighted": (
            sentence_vectors(train_tokens, vectors, train_tfidf, tfidf_vocabulary),
            sentence_vectors(test_tokens, vectors, test_tfidf, tfidf_vocabulary),
        ),
    }

    for representation_name, (x_train, x_test) in representations.items():
        classifier = make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(100,),
                activation="relu",
                solver="adam",
                max_iter=100,
                early_stopping=True,
                n_iter_no_change=5,
                random_state=42,
            ),
        )
        classifier.fit(x_train, y_train)
        predictions = classifier.predict(x_test)
        network_results.append({
            "embedding": model_name,
            "representation": representation_name,
            "macro_f1": f1_score(y_test, predictions, average="macro"),
        })

    del vectors, representations
    gc.collect()

network_results_df = pd.DataFrame(network_results)
network_results_df.to_csv(OUTPUT_DIR / "q3_macro_f1.csv", index=False)
display(network_results_df)


,embedding,representation,macro_f1
0,Word2Vec,Average,0.795918
1,Word2Vec,TF-IDF weighted,0.779999
2,GloVe,Average,0.772982
3,GloVe,TF-IDF weighted,0.743991
4,FastText,Average,0.810882
5,FastText,TF-IDF weighted,0.762998


In [5]:
analogies = []
with ANALOGY_PATH.open(encoding="utf-8") as analogy_file:
    category = ""
    for line in analogy_file:
        fields = line.strip().lower().split()
        if len(fields) == 2 and fields[0] == ":":
            category = fields[1]
        elif len(fields) == 4:
            analogies.append((category, *fields))

def analogy_accuracy(keyed_vectors, examples):
    correct = 0
    total = 0
    for _, first, second, third, answer in examples:
        if any(word not in keyed_vectors.key_to_index for word in (first, second, third, answer)):
            continue
        predicted = keyed_vectors.most_similar(
            positive=[third, second],
            negative=[first],
            topn=1,
        )[0][0]
        total += 1
        correct += predicted == answer
    return correct, total

analogy_results = []
for model_name in EMBEDDING_PATHS:
    vectors = load_embeddings(model_name)
    correct, total = analogy_accuracy(vectors, analogies)
    analogy_results.append({
        "embedding": model_name,
        "correct": correct,
        "evaluated": total,
        "accuracy": correct / total if total else np.nan,
    })
    del vectors
    gc.collect()

analogy_accuracy_df = pd.DataFrame(analogy_results)
analogy_accuracy_df.to_csv(OUTPUT_DIR / "q2_analogy_accuracy.csv", index=False)
display(analogy_accuracy_df)


,embedding,correct,evaluated,accuracy
0,Word2Vec,7581,13705,0.553156
1,GloVe,14020,19544,0.717356
2,FastText,11222,15845,0.708236
